# Surgical Video Assistant Training Notebook

Notebook này dùng để train và trình bày project **Surgical Frame/Image VQA + Phase/Tool/Action Recognition**.

Mục tiêu:
- Clone repo từ GitHub.
- Cài dependencies cho VM/Colab.
- Chuẩn hóa dataset JSONL.
- Chạy baseline zero-shot/few-shot hoặc LoRA/QLoRA SFT.
- Có log, tqdm, checkpoint, chart loss, metrics, và artifact để show với Thầy.

> Research/education only. Not clinical decision support.

## 1. Clone Repo

Sau khi repo được push lên GitHub, chỉ cần sửa `GITHUB_REPO_URL` thành URL repo của bạn rồi chạy cell này.

In [ ]:
# One-cell clone for notebook/Colab/VM
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/surgical-video-assistant.git"  # TODO: replace after pushing repo
PROJECT_DIR = "/content/surgical-video-assistant"

import os, subprocess, sys
from pathlib import Path

if not Path(PROJECT_DIR).exists():
    subprocess.run(["git", "clone", GITHUB_REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

## 2. Install Dependencies

Base dependencies đủ để parse/evaluate/demo. Training dependencies dùng cho GPU VM với LoRA/QLoRA.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
print("Dependencies installed")

## 3. Check GPU and Runtime

In [ ]:
import os, sys, platform, subprocess

print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found. Use a GPU VM/Colab runtime for training.")

## 4. Mount/Upload Dataset

Đặt raw SurgMLLMBench vào `data/raw/SurgMLLMBench`.

Nếu dùng Google Drive trên Colab, mount Drive rồi copy/symlink dataset vào đường dẫn này.

In [ ]:
from pathlib import Path

RAW_DIR = Path("data/raw/SurgMLLMBench")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Expected raw dataset path:", RAW_DIR.resolve())
print("Put SurgMLLMBench raw JSON/JSONL + image files here before prepare step.")

## 5. Prepare Dataset

Parser đọc raw JSON/JSONL trực tiếp, không phụ thuộc Hugging Face dataset viewer.

In [ ]:
import subprocess, sys
from pathlib import Path

raw_has_files = any(RAW_DIR.rglob("*.json")) or any(RAW_DIR.rglob("*.jsonl"))
if raw_has_files:
    subprocess.run([
        sys.executable, "-m", "src.data.prepare",
        "--dataset", "surgmllmbench",
        "--raw", str(RAW_DIR),
        "--out", str(PROCESSED_DIR),
    ], check=True)
else:
    print("No raw JSON/JSONL found. Using example dataset for smoke test.")
    print("For real training, upload SurgMLLMBench into", RAW_DIR)

## 6. Inspect Dataset Summary

In [ ]:
import json
from pathlib import Path

summary_path = Path("data/processed/dataset_summary.json")
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2, ensure_ascii=False)[:5000])
else:
    print("No dataset_summary.json yet. Prepare real data first.")

## 7. Smoke-Test Inference + Evaluation

Mock provider giúp kiểm tra pipeline trước khi chạy model thật.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "src.models.run_inference", "--config", "configs/mock_zero_shot.yaml"], check=True)
subprocess.run([
    sys.executable, "-m", "src.eval.evaluate",
    "--pred", "reports/predictions_mock.jsonl",
    "--out", "reports/metrics_mock.json",
], check=True)

## 8. LoRA/QLoRA Training

Chạy cell này trên VM GPU. Checkpoint sẽ lưu vào `checkpoints/gemma4-surgical-frame-lora` theo `save_steps` trong config.

In [ ]:
import subprocess, sys, time
from pathlib import Path

train_jsonl = Path("data/processed/processed_dataset.jsonl")
if not train_jsonl.exists():
    print("Real processed dataset not found:", train_jsonl)
    print("Run prepare step with real SurgMLLMBench before training.")
else:
    start = time.time()
    subprocess.run([sys.executable, "-m", "src.models.train_sft", "--config", "configs/gemma4_lora_sft.yaml"], check=True)
    print(f"Training finished in {(time.time() - start) / 60:.2f} minutes")

## 9. Plot Training Logs

Trainer logs thường nằm trong checkpoint/log history hoặc TensorBoard files tùy Transformers version. Cell này cố gắng đọc `trainer_state.json` từ checkpoint mới nhất và lưu chart vào `reports/figures`.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

FIG_DIR = Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
ckpt_root = Path("checkpoints/gemma4-surgical-frame-lora")
state_files = sorted(ckpt_root.glob("checkpoint-*/trainer_state.json"))
if not state_files and (ckpt_root / "trainer_state.json").exists():
    state_files = [ckpt_root / "trainer_state.json"]

if not state_files:
    print("No trainer_state.json found yet. Train first.")
else:
    state_path = state_files[-1]
    state = json.loads(state_path.read_text())
    logs = state.get("log_history", [])
    train = [(x.get("step"), x.get("loss")) for x in logs if x.get("loss") is not None]
    evals = [(x.get("step"), x.get("eval_loss")) for x in logs if x.get("eval_loss") is not None]

    plt.figure(figsize=(8, 5))
    if train:
        plt.plot([x for x, _ in train], [y for _, y in train], label="train loss")
    if evals:
        plt.plot([x for x, _ in evals], [y for _, y in evals], label="eval loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Gemma Surgical Frame LoRA Training")
    plt.grid(True, alpha=0.3)
    plt.legend()
    out = FIG_DIR / "training_loss.png"
    plt.savefig(out, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved chart:", out)

## 10. Evaluate Real Model Outputs

Sau khi chạy inference bằng config Gemma thật, cell này tạo metrics JSON và chart tóm tắt task-level.

In [ ]:
import subprocess, sys, json
from pathlib import Path
import matplotlib.pyplot as plt

pred_path = Path("reports/predictions_gemma4_zero_shot.jsonl")
metrics_path = Path("reports/metrics_gemma4_zero_shot.json")

if pred_path.exists():
    subprocess.run([sys.executable, "-m", "src.eval.evaluate", "--pred", str(pred_path), "--out", str(metrics_path)], check=True)
    metrics = json.loads(metrics_path.read_text())
    task_names = []
    task_scores = []
    for task, values in metrics.get("by_task", {}).items():
        task_names.append(task)
        task_scores.append(values.get("accuracy", values.get("exact_match", values.get("micro_f1", 0))))

    if task_names:
        plt.figure(figsize=(8, 4))
        plt.bar(task_names, task_scores)
        plt.ylim(0, 1)
        plt.ylabel("Score")
        plt.title("Task-level Evaluation Summary")
        plt.xticks(rotation=25, ha="right")
        out = Path("reports/figures/task_scores.png")
        plt.savefig(out, dpi=160, bbox_inches="tight")
        plt.show()
        print("Saved chart:", out)
else:
    print("Prediction file not found yet:", pred_path)
    print("Run Gemma inference first, e.g. python -m src.models.run_inference --config configs/gemma4_zero_shot.yaml")

## 11. Package Presentation Artifacts

Tạo danh sách artifact để đưa vào slide/report.

In [ ]:
from pathlib import Path

for path in [
    "reports/metrics_mock.json",
    "reports/metrics_gemma4_zero_shot.json",
    "reports/figures/training_loss.png",
    "reports/figures/task_scores.png",
    "checkpoints/gemma4-surgical-frame-lora",
]:
    p = Path(path)
    print("FOUND" if p.exists() else "MISSING", "-", path)